# 实践 14.1  DH测交性状预测与组合潜力排序

《统计遗传与智能育种学》第十四章 · 电子资源 `14_dh_top_potential.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：14_dh_top_potential.ipynb。Notebook保留Gene_encode()、Generate_genotype_DH()、Pretrait_test()、Top_target()和DHepo()等R函数入口名称，但资源中只有函数名。可运行Python部分构造6个双亲组合，每个组合产生80个DH，每个DH再与2个测验种逐一测交，得到960条0/1/2剂量测交基因型。由于模拟保留真实标记效应，除了组合的优良DH数量，还能直接验证三性状GS的真值相关和目标排序是否可靠。

本章Python实验采用独立的可运行简化实现，不替代未公开的R函数或TOP原始优化器。为保证遗传编码一致，模拟双亲和测验种在每个示例标记上均为纯合0/2；DH逐位点抽取一个亲本等位基因并加倍，DH×测验种的F₁剂量为(DH+测验种)/2，因此只可取0/1/2。这一模拟生成器把160个标记当作独立位点，未模拟染色体连锁和真实重组。

R自定义函数名清单（接口名称示意，不是可执行代码）

Gene_encode；Generate_genotype_DH；Pretrait_test；Top_target；DHepo。

## 输入、完整代码、实际输出与解释

输入与可复现条件：SEED=1401；360个训练样本、160个标记、3个性状；6个双亲组合×80个DH×2个测验种；不需外部数据文件。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1401
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1401


In [2]:
from sklearn.linear_model import Ridge

m, ntrain, ncross, ndh, ntraits = 160, 360, 6, 80, 3
p = rng.uniform(0.10, 0.50, m)
Xtr = rng.binomial(2, p, (ntrain, m)).astype(float)

B = rng.normal(0, 1, (m, ntraits))
g0 = Xtr @ B
desired_sd = np.array([1.2, 3.0, 5.0])
B *= desired_sd / g0.std(axis=0)
intercept = np.array([8.0, 40.0, 110.0])
resid_sd = np.array([1.0, 2.5, 4.0])
Y = intercept + Xtr @ B + rng.normal(0, resid_sd, (ntrain, ntraits))

models = [Ridge(alpha=35).fit(Xtr, Y[:, j]) for j in range(ntraits)]
trait_names = ['yield', 'protein', 'maturity']
print('training samples/markers', ntrain, m)
print('training trait means', np.round(Y.mean(axis=0), 3))


training samples/markers 360 160
training trait means [  8.191  39.941 121.44 ]


In [3]:
parents = 2.0 * rng.binomial(1, p, (ncross, 2, m))
testers = 2.0 * rng.binomial(1, p, (2, m))

records = []
for cross in range(ncross):
    for dh_id in range(ndh):
        choose_parent = rng.integers(0, 2, m)
        dh = parents[cross, choose_parent, np.arange(m)]
        assert np.isin(dh, [0.0, 2.0]).all()
        for tester_id, tester in enumerate(testers):
            testcross = (dh + tester) / 2.0
            assert np.isin(testcross, [0.0, 1.0, 2.0]).all()
            pred = np.array([model.predict(testcross.reshape(1, -1))[0] for model in models])
            true_value = intercept + testcross @ B
            records.append((cross, dh_id, tester_id, *pred, *true_value))

cols = ['cross', 'DH', 'tester'] + [f'pred_{x}' for x in trait_names] + [f'true_{x}' for x in trait_names]
df = pd.DataFrame(records, columns=cols)
print('DH/testcross records', ncross * ndh, len(df))
for trait in trait_names:
    r = np.corrcoef(df[f'pred_{trait}'], df[f'true_{trait}'])[0, 1]
    print(trait, 'GS r to true genetic value', round(float(r), 3))


DH/testcross records

 480 960
yield GS r to true genetic value 0.821
protein GS r to true genetic value 0.889
maturity GS r to true genetic value 0.859


In [4]:
pred_cols = [f'pred_{x}' for x in trait_names]
true_cols = [f'true_{x}' for x in trait_names]
check = df[pred_cols].median().to_numpy()
target = check.copy()
target[1] *= 1.05  # 示例目标：在对照基础上把性状2提高5%
scale = Y.std(axis=0)
weights = np.array([0.4, 0.4, 0.2])

pred_delta = np.abs((df[pred_cols].to_numpy() - target) / scale)
true_delta = np.abs((df[true_cols].to_numpy() - target) / scale)
df['pred_similarity'] = np.exp(-(pred_delta * weights).sum(axis=1))
df['true_similarity'] = np.exp(-(true_delta * weights).sum(axis=1))

cut_pred = df['pred_similarity'].quantile(0.80)
cut_true = df['true_similarity'].quantile(0.80)
df['pred_favorable'] = df['pred_similarity'] >= cut_pred
df['true_favorable'] = df['true_similarity'] >= cut_true
precision = (df.pred_favorable & df.true_favorable).sum() / df.pred_favorable.sum()
rank_r = np.corrcoef(df.pred_similarity.rank(), df.true_similarity.rank())[0, 1]

summary = df.groupby('cross').apply(
    lambda g: pd.Series({
        'favorable_testcross': int(g.pred_favorable.sum()),
        'favorable_DH': int(g.loc[g.pred_favorable, 'DH'].nunique()),
        'favorable_DH_prop': g.loc[g.pred_favorable, 'DH'].nunique() / ndh,
        'mean_pred_similarity': g.pred_similarity.mean(),
        'mean_true_similarity': g.true_similarity.mean()
    }), include_groups=False
)
print('check', np.round(check, 3), 'target', np.round(target, 3), 'weights', weights)
print('pred-vs-true similarity rank r', round(float(rank_r), 3))
print('top20% predicted favorable precision against true top20%', round(float(precision), 3))
print(summary.sort_values('favorable_DH', ascending=False).round(3).to_string())


check [  8.316  40.339 123.983] target [  8.316  42.356 123.983] weights [0.4 0.4 0.2]
pred-vs-true similarity rank r 0.76
top20% predicted favorable precision against true top20% 0.589
       favorable_testcross  favorable_DH  favorable_DH_prop  mean_pred_similarity  mean_true_similarity
cross                                                                                                  
3                     71.0          49.0              0.612                 0.741                 0.704
5                     45.0          35.0              0.438                 0.681                 0.638
2                     23.0          21.0              0.262                 0.627                 0.502
0                     24.0          18.0              0.225                 0.639                 0.549
1                     23.0          17.0              0.212                 0.630                 0.541
4                      6.0           5.0              0.062                 0.566     

结果解释：三个性状的预测值与模拟遗传值的Pearson相关依次为0.821、0.889和0.859。预测相似度与真实相似度的秩相关为0.760；预测前20%的测交材料中，58.9%也位于真值前20%。组合3产生49个至少有1个测验种后代落入预测前20%的DH，而组合4只有5个。本例以预测和模拟真值的入选重合率评价排序。

可修改练习：把蛋白目标从候选预测中位数上调5%改为+10%，再把性状权重由[0.4,0.4,0.2]改为[0.2,0.6,0.2]，比较各组合favorable_DH排名；不得使用最终真值反向挑选权重。